In [ ]:
pip install sentencepiece

In [7]:
# ============================================================
# CELL 1 — Imports + Reproducibility
# Purpose:
#   - Import all dependencies
#   - Set deterministic seeds
# Inputs:
#   - None
# Outputs:
#   - Seeded RNG state
# ============================================================

import os
import re
import ast
import math
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
import yake


In [8]:
# ============================================================
# CELL 2 — Global Config
# Purpose:
#   - Centralize paths and constants
# Inputs:
#   - None
# Outputs:
#   - Config variables
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR = r"/teamspace/studios/this_studio/datasets"
EMB_DIR  = r"/teamspace/studios/this_studio/embeddings"
CKPT_DIR = r"/teamspace/studios/this_studio/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_T = 100  # max total sequence length in data
# history will use up to t-1, so effective history length is <= MAX_T-1

# Task sets (must match main_df["task"])
TASK_NEXT_USER = "next_user_prediction"
TASK_NEXT_ITEM = "next_item_prediction"
TASK_DIALOGUE  = "dialogue_generation"
TASK_SUMMARY   = "personalized_summarization"

KP_TASKS = {TASK_DIALOGUE, TASK_SUMMARY}


In [ ]:
# ============================================================
# CELL 3 — Load CSVs + Embeddings
# Purpose:
#   - Load all tables and embedding dicts
# Inputs:
#   - Config paths
# Outputs:
#   - main_df, dialog_df, docs_df, resp_df, summ_df
#   - emb_dialog, emb_docs, emb_resp, emb_summ
# ============================================================

def load_embeddings(file_path: str):
    with open(file_path, "rb") as f:
        return pickle.load(f)

main_df   = pd.read_csv(os.path.join(DATA_DIR, "main_mixed_df.csv"))
dialog_df = pd.read_csv(os.path.join(DATA_DIR, "dialogue.csv"))
docs_df   = pd.read_csv(os.path.join(DATA_DIR, "documents.csv"))
resp_df   = pd.read_csv(os.path.join(DATA_DIR, "response.csv"))
summ_df   = pd.read_csv(os.path.join(DATA_DIR, "summaries.csv"))

emb_dialog = load_embeddings(os.path.join(EMB_DIR, "dialog_embeddings.pkl"))
emb_docs   = load_embeddings(os.path.join(EMB_DIR, "doc_embeddings.pkl"))
emb_resp   = load_embeddings(os.path.join(EMB_DIR, "response_embeddings.pkl"))
emb_summ   = load_embeddings(os.path.join(EMB_DIR, "summ_embeddings.pkl"))

print("Loaded main_df columns:", list(main_df.columns))
print("Rows:", len(main_df))
print("Tasks:", sorted(main_df["task"].dropna().astype(str).unique().tolist()))


In [10]:
# ============================================================
# CELL 4 — Core Helpers (parsing, null checks, canonical IDs)
# Purpose:
#   - Robust parsing for serialized sequences
#   - Canonical ID normalization (dl-#, d-#, r-#, s-#)
# Inputs:
#   - raw strings from CSV
# Outputs:
#   - utility functions
# ============================================================

CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def is_nullish(x) -> bool:
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "null", "none", "nan"}

def normalize_id(x) -> str:
    return str(x).strip()

def to_canonical(prefix: str, raw_id) -> str:
    s = str(raw_id).strip()
    if CANON_RE.match(s):
        return s
    return f"{prefix}-{s}"

def parse_serialized_sequence(x):
    """
    Accepts:
      - python list
      - string like '["a","b"]'
      - string like '[a,b]'
    Returns:
      - list[str]
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]

    s = str(x).strip()
    if not (s.startswith("[") and s.endswith("]")):
        return []

    # Try safe literal_eval first
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(z).strip().strip('"').strip("'") for z in v if str(z).strip()]
    except Exception:
        pass

    # Fallback: comma split
    body = s[1:-1].strip()
    if not body:
        return []
    items = [t.strip().strip('"').strip("'") for t in body.split(",")]
    return [t for t in items if t]


In [ ]:
# ============================================================
# CELL 5 — Normalize split + Create Train/Test
# Purpose:
#   - Standardize split column and create train_df/test_df
# Inputs:
#   - main_df
# Outputs:
#   - train_df, test_df
# ============================================================

if "split" in main_df.columns:
    main_df["split"] = main_df["split"].astype(str).str.strip().str.lower()
else:
    raise ValueError("main_df must contain a 'split' column with values like train/test")

train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df), "| Test size:", len(test_df))


In [ ]:
# ============================================================
# CELL 6 — Build ACTION vocabulary (global)
# Purpose:
#   - Build actions2idx from ALL rows (shared input feature)
# Inputs:
#   - main_df["interactions"]
# Outputs:
#   - actions2idx, idx2action
# ============================================================

all_actions = set()
for seq in main_df["interactions"].dropna():
    for a in parse_serialized_sequence(seq):
        if not is_nullish(a):
            all_actions.add(a)

actions2idx = {a: i for i, a in enumerate(sorted(all_actions))}
idx2action  = {i: a for a, i in actions2idx.items()}

print("Actions vocab:", len(actions2idx))


In [ ]:
# ============================================================
# CELL 7 — Build NODE vocabulary (global, for sequences only)
# Purpose:
#   - Build node2idx from ALL text_nodes sequences
#   - This is NOT the "item vocab"; it's for encoding sequences
# Inputs:
#   - main_df["text_nodes"]
# Outputs:
#   - node2idx, idx2node
# ============================================================

node_ids = set()
for seq in main_df["text_nodes"].dropna():
    for nid in parse_serialized_sequence(seq):
        if not is_nullish(nid):
            node_ids.add(normalize_id(nid))

node2idx = {nid: i for i, nid in enumerate(sorted(node_ids))}
idx2node = {i: nid for nid, i in node2idx.items()}

print("Node vocab (sequence nodes):", len(node2idx))


In [ ]:
# ============================================================
# CELL 8 — Build USER vocabulary (ONLY for next_user_prediction rows)
# Purpose:
#   - user2idx is built using only rows with task == next_user_prediction
# Inputs:
#   - main_df filtered by task
# Outputs:
#   - user2idx, idx2user
# ============================================================

user_rows = main_df[main_df["task"].astype(str).str.strip() == TASK_NEXT_USER]
unique_users = sorted(user_rows["userid"].dropna().astype(str).unique().tolist())

user2idx = {u: i for i, u in enumerate(unique_users)}
idx2user = {i: u for u, i in user2idx.items()}

print("User vocab (next_user_prediction only):", len(user2idx))


In [ ]:
# ============================================================
# CELL 9 — Build ITEM vocabulary (ONLY for next_item_prediction using query_text_id)
# Purpose:
#   - item2idx is built using only query_text_id where task == next_item_prediction
# Inputs:
#   - main_df filtered by task
# Outputs:
#   - item2idx, idx2item
# ============================================================

item_rows = main_df[main_df["task"].astype(str).str.strip() == TASK_NEXT_ITEM]

if "query_text_id" not in main_df.columns:
    raise ValueError("main_df must contain 'query_text_id'")

unique_items = (
    item_rows["query_text_id"]
    .dropna()
    .astype(str)
    .map(normalize_id)
    .unique()
    .tolist()
)
unique_items = sorted(unique_items)

item2idx = {it: i for i, it in enumerate(unique_items)}
idx2item = {i: it for it, i in item2idx.items()}

print("Item vocab (next_item_prediction query_text_id only):", len(item2idx))


In [ ]:
# ============================================================
# CELL 10 — Build TASK vocabulary (global)
# Purpose:
#   - Map all tasks to indices (task prediction head)
# Inputs:
#   - main_df["task"]
# Outputs:
#   - task2idx, idx2task, main_df["task_id"]
# ============================================================

if "task" not in main_df.columns:
    raise ValueError("main_df must contain 'task'")

task_names = sorted(main_df["task"].dropna().astype(str).unique().tolist())
task2idx = {t: i for i, t in enumerate(task_names)}
idx2task = {i: t for t, i in task2idx.items()}

main_df["task_id"] = main_df["task"].astype(str).map(task2idx)
if main_df["task_id"].isna().sum() != 0:
    raise ValueError("Found unmapped task(s) in main_df")

train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

n_tasks = len(task2idx)
print("Tasks:", n_tasks, task2idx)


In [ ]:
# ============================================================
# CELL 11 — Build id2text (canonical IDs, no double prefix)
# Purpose:
#   - Create canonical_id -> text mapping for all text sources
# Inputs:
#   - dialog_df, docs_df, resp_df, summ_df
# Outputs:
#   - id2text dict
# ============================================================

def build_id2text(dialog_df, docs_df, resp_df, summ_df):
    id2text = {}

    # Dialogue
    if "dialogue_id" in dialog_df.columns:
        for _, r in dialog_df.dropna(subset=["dialogue_id"]).iterrows():
            cid = to_canonical("dl", r["dialogue_id"])
            id2text[cid] = str(r.get("dialogue_text", ""))

    # Docs
    if "NewsID" in docs_df.columns:
        for _, r in docs_df.dropna(subset=["NewsID"]).iterrows():
            cid = to_canonical("d", r["NewsID"])
            nb = str(r.get("NewsBody", "")).strip()
            hl = str(r.get("Headline", "")).strip()
            id2text[cid] = nb if nb else hl

    # Responses
    if "response_id" in resp_df.columns:
        for _, r in resp_df.dropna(subset=["response_id"]).iterrows():
            cid = to_canonical("r", r["response_id"])
            id2text[cid] = str(r.get("response_text", ""))

    # Summaries
    if "SummID" in summ_df.columns:
        for _, r in summ_df.dropna(subset=["SummID"]).iterrows():
            cid = to_canonical("s", r["SummID"])
            id2text[cid] = str(r.get("Summary", ""))

    return id2text

id2text = build_id2text(dialog_df, docs_df, resp_df, summ_df)
print("id2text size:", len(id2text))
print("sample keys:", list(id2text.keys())[:10])


In [12]:
# # ============================================================
# # CELL 12 — Keyphrase extraction setup (YAKE)
# # Purpose:
# #   - Configure YAKE extraction
# # Inputs:
# #   - None
# # Outputs:
# #   - kw_extractor
# # ============================================================

# MAX_KP_PER_TEXT = 50

# YAKE_CONFIG = dict(
#     lan="en",
#     n=3,
#     dedupLim=0.9,
#     dedupFunc="seqm",
#     windowsSize=1,
#     top=MAX_KP_PER_TEXT
# )

# VALID_PHRASE_REGEX = re.compile(r"^[a-z0-9]+( [a-z0-9]+)*$")

# kw_extractor = yake.KeywordExtractor(**YAKE_CONFIG)


In [ ]:
# # ============================================================
# # CELL 13 — Build id2phrases ONLY from KP_TASKS using query_text_id & target_text_id
# # Purpose:
# #   - For dialogue_generation + personalized_summarization:
# #       collect unique IDs from query_text_id and target_text_id
# #       extract YAKE keyphrases only for those IDs
# # Inputs:
# #   - main_df, id2text
# # Outputs:
# #   - id2phrases, kp2idx, idx2kp
# # ============================================================

# for col in ["query_text_id", "target_text_id"]:
#     if col not in main_df.columns:
#         raise ValueError(f"Missing column: {col}")

# kp_rows = main_df[main_df["task"].astype(str).str.strip().isin(KP_TASKS)]

# unique_kp_text_ids = set()
# unique_kp_text_ids.update(
#     kp_rows["query_text_id"].dropna().astype(str).map(normalize_id).unique().tolist()
# )
# unique_kp_text_ids.update(
#     kp_rows["target_text_id"].dropna().astype(str).map(normalize_id).unique().tolist()
# )

# print("Unique text IDs for KP extraction (KP_TASKS only):", len(unique_kp_text_ids))

# id2text_unique = {}
# for tid in unique_kp_text_ids:
#     if tid in id2text:
#         txt = str(id2text[tid]).strip()
#         if txt:
#             id2text_unique[tid] = txt

# print("Texts actually processed for KPs:", len(id2text_unique))

# id2phrases = {}
# kp_vocab = set()

# for tid, text in tqdm(id2text_unique.items(), desc="Extracting YAKE KPs", total=len(id2text_unique)):
#     try:
#         raw_kps = kw_extractor.extract_keywords(text)
#     except Exception:
#         id2phrases[tid] = []
#         continue

#     clean = []
#     seen = set()
#     for phrase, score in raw_kps:
#         p = str(phrase).lower().strip()
#         if p in seen:
#             continue
#         if not VALID_PHRASE_REGEX.match(p):
#             continue
#         if len(p) < 3:
#             continue

#         clean.append(p)
#         seen.add(p)
#         if len(clean) >= MAX_KP_PER_TEXT:
#             break

#     id2phrases[tid] = clean
#     kp_vocab.update(clean)

# kp_vocab = sorted(kp_vocab)
# kp2idx = {kp: i for i, kp in enumerate(kp_vocab)}
# idx2kp = {i: kp for kp, i in kp2idx.items()}

# print("KP vocab size:", len(kp2idx))
# print("Texts with 0 KPs:", sum(1 for v in id2phrases.values() if len(v) == 0), "/", len(id2phrases))


In [ ]:
# # ============================================================
# # CELL — Save Keyphrase Artifacts
# # Purpose:
# #   - Persist id2phrases and kp2idx to disk
# # Outputs:
# #   - kp_id2phrases.pkl
# #   - kp_vocab.pkl
# # ============================================================

# import pickle
# import os

# KP_DIR = "/teamspace/studios/this_studio/keyphrases"
# os.makedirs(KP_DIR, exist_ok=True)

# id2phrases_path = os.path.join(KP_DIR, "id2phrases.pkl")
# kp_vocab_path   = os.path.join(KP_DIR, "kp_vocab.pkl")

# with open(id2phrases_path, "wb") as f:
#     pickle.dump(id2phrases, f)

# with open(kp_vocab_path, "wb") as f:
#     pickle.dump(kp2idx, f)

# print("Saved:")
# print("  id2phrases →", id2phrases_path)
# print("  kp2idx     →", kp_vocab_path)


In [ ]:
# ============================================================
# CELL — Load Keyphrase Artifacts
# Purpose:
#   - Reload id2phrases and kp2idx exactly as before
# Outputs:
#   - id2phrases
#   - kp2idx
#   - idx2kp (reconstructed)
# ============================================================

import pickle

KP_DIR = "/teamspace/studios/this_studio/keyphrases"

id2phrases_path = os.path.join(KP_DIR, "id2phrases.pkl")
kp_vocab_path   = os.path.join(KP_DIR, "kp_vocab.pkl")

with open(id2phrases_path, "rb") as f:
    id2phrases = pickle.load(f)

with open(kp_vocab_path, "rb") as f:
    kp2idx = pickle.load(f)

idx2kp = {i: kp for kp, i in kp2idx.items()}

print("Loaded:")
print("  id2phrases size:", len(id2phrases))
print("  kp vocab size :", len(kp2idx))


In [ ]:
from collections import defaultdict

total_rows = len(main_df)

# global counters
rows_with_target = 0
rows_with_valid_kp = 0
rows_zero_kp = 0
rows_missing_text = 0

# task-wise stats
task_stats = defaultdict(lambda: {
    "rows": 0,
    "kp_expected": 0,
    "kp_present": 0,
    "kp_zero": 0,
    "kp_missing": 0,
})

for _, row in main_df.iterrows():
    task = str(row["task"]).strip()
    task_stats[task]["rows"] += 1

    tid = row.get("target_text_id", None)
    if tid is None or str(tid).strip() == "":
        continue  # no KP expected

    tid = normalize_id(tid)
    rows_with_target += 1
    task_stats[task]["kp_expected"] += 1

    if tid not in id2phrases:
        rows_missing_text += 1
        task_stats[task]["kp_missing"] += 1
        continue

    if len(id2phrases[tid]) == 0:
        rows_zero_kp += 1
        task_stats[task]["kp_zero"] += 1
        continue

    rows_with_valid_kp += 1
    task_stats[task]["kp_present"] += 1


# ----------------------------
# GLOBAL SUMMARY
# ----------------------------
print("\n========== GLOBAL KP COVERAGE ==========")
print(f"Total rows                : {total_rows}")
print(f"Rows with target_text_id  : {rows_with_target}")
print(f"Rows with VALID KP        : {rows_with_valid_kp}")
print(f"Rows with ZERO KP         : {rows_zero_kp}")
print(f"Rows with MISSING text    : {rows_missing_text}")

if rows_with_target > 0:
    print(
        f"Effective KP supervision : "
        f"{rows_with_valid_kp / rows_with_target:.3f}"
    )

# ----------------------------
# TASK-WISE SUMMARY
# ----------------------------
print("\n========== TASK-WISE KP COVERAGE ==========")

for task, s in sorted(task_stats.items()):
    if s["kp_expected"] == 0:
        continue

    eff = s["kp_present"] / max(s["kp_expected"], 1)

    print(
        f"{task:30s} | "
        f"rows={s['rows']:6d} | "
        f"kp_expected={s['kp_expected']:6d} | "
        f"kp_valid={s['kp_present']:6d} | "
        f"kp_zero={s['kp_zero']:6d} | "
        f"kp_missing={s['kp_missing']:6d} | "
        f"eff={eff:.3f}"
    )


In [ ]:
def has_valid_kp(row):
    tid = row.get("target_text_id", None)
    if tid is None or str(tid).strip() == "":
        return False

    tid = normalize_id(tid)

    if tid not in id2phrases:
        return False

    if len(id2phrases[tid]) == 0:
        return False

    return True


before = len(main_df)

main_df = main_df[main_df.apply(has_valid_kp, axis=1)].reset_index(drop=True)

after = len(main_df)

print("Dropped rows:", before - after)
print("Remaining rows:", after)


In [ ]:
main_df

In [22]:
# ============================================================
# CELL 14 — KP supervision helpers (ONLY for KP_TASKS)
# Purpose:
#   - Decide kp supervision id (target_text_id) for KP tasks only
# Inputs:
#   - row
# Outputs:
#   - kp_id or None, multi-hot KP target
# ============================================================

def get_kp_supervision_id(row):
    task = str(row["task"]).strip()
    if task in KP_TASKS:
        tid = row.get("target_text_id", None)
        if not is_nullish(tid):
            return normalize_id(tid)
    return None

def build_kp_target_from_id(kp_id, id2phrases, kp2idx, device):
    phrases = id2phrases.get(kp_id, [])
    y = torch.zeros(len(kp2idx), device=device)
    for p in phrases:
        j = kp2idx.get(p, None)
        if j is not None:
            y[j] = 1.0
    return y, phrases


In [23]:
# ============================================================
# CELL 15 — Embedding Resolver (canonical IDs)
# Purpose:
#   - Resolve embeddings by canonical id across dl/d/r/s
# Inputs:
#   - emb_* dicts
# Outputs:
#   - emb_resolver.get(cid) -> np.array(768,)
# ============================================================

class EmbLookup:
    @staticmethod
    def lookup(emb_dict, key):
        if key in emb_dict:
            return emb_dict[key]
        s = str(key)
        if s in emb_dict:
            return emb_dict[s]
        nums = re.findall(r"\d+", s)
        if nums:
            n = nums[-1]
            if n in emb_dict:
                return emb_dict[n]
            try:
                ni = int(n)
                if ni in emb_dict:
                    return emb_dict[ni]
            except Exception:
                pass
        return None

class CanonicalEmbeddingResolver:
    def __init__(self, emb_dialog, emb_docs, emb_resp, emb_summ, d=768):
        self.emb_dialog = emb_dialog
        self.emb_docs   = emb_docs
        self.emb_resp   = emb_resp
        self.emb_summ   = emb_summ
        self.d = d

    def get(self, cid: str) -> np.ndarray:
        cid = normalize_id(cid)
        prefix = cid.split("-")[0].lower()

        if prefix == "dl":
            v = EmbLookup.lookup(self.emb_dialog, cid)
        elif prefix == "d":
            v = EmbLookup.lookup(self.emb_docs, cid)
        elif prefix == "r":
            v = EmbLookup.lookup(self.emb_resp, cid)
        elif prefix == "s":
            v = EmbLookup.lookup(self.emb_summ, cid)
        else:
            v = None

        if v is None:
            return np.zeros((self.d,), dtype=np.float32)

        v = np.asarray(v, dtype=np.float32)
        if v.shape[-1] != self.d:
            raise ValueError(f"Embedding dim mismatch for {cid}: got {v.shape}, expected ({self.d},)")
        return v

emb_resolver = CanonicalEmbeddingResolver(emb_dialog, emb_docs, emb_resp, emb_summ, d=768)


In [70]:
# ============================================================
# CELL — UniHisSSM Encoder + Per-step losses (ADD-ONLY)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class UniHisSSMEncoder(nn.Module):
    def __init__(
        self,
        d_model,
        d_state,
        n_tasks,
        n_actions,      # NEW
        known_task=False
    ):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.n_tasks = n_tasks
        self.known_task = known_task

        # ----- SSM eigen params -----
        def init_eig():
            Q = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
            Lambda = nn.Parameter(torch.empty(d_state).uniform_(-1.0, -0.1))
            return Q, Lambda

        self.Q_L, self.Lambda_L = init_eig()
        self.Q_S, self.Lambda_S = init_eig()
        self.Q_E, self.Lambda_E = init_eig()

        self.B = nn.Parameter(torch.randn(d_state, d_model))

        self.tau_S = nn.Parameter(torch.tensor(1.0))
        self.tau_E = nn.Parameter(torch.tensor(1.0))

        # ----- ACTION -> mood -----
        self.MLP_mood = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )
        self.W_a = nn.Linear(d_model, d_state)

        # ----- Fusion -----
        self.Mixer_L = nn.Linear(d_state, d_state)
        self.Mixer_S = nn.Linear(d_state, d_state)
        self.Mixer_E = nn.Linear(d_state, d_state)

        self.W_fuse = nn.Linear(3 * d_state, 3)
        self.Mixer_Fuse = nn.Linear(d_state, d_state)

        # ----- Task conditioning -----
        self.Embed_Q = nn.Linear(d_model, d_state)
        self.W_task  = nn.Linear(2 * d_state, n_tasks)

        self.W_hyp = nn.Linear(2 * d_state, d_state)
        self.W_theta = nn.ModuleList([nn.Linear(d_state, 2 * d_state) for _ in range(n_tasks)])

        # =====================================================
        # NEW: PER-STEP PREDICTION HEADS (ADD ONLY)
        # =====================================================

        self.ActionHead = nn.Linear(3 * d_state, n_actions)   # action_t
        self.NodeHead   = nn.Linear(d_state, d_model)        # reconstruct e_t

        self.ce = nn.CrossEntropyLoss()
        self.mse = nn.MSELoss(reduction="none")

    def make_A(self, Q, Lambda_diag):
        Q_inv = torch.linalg.inv(Q)
        return Q @ torch.diag(Lambda_diag) @ Q_inv

    def forward(
        self,
        e_hist,          # (B, Th, d_model)
        a_hist_emb,      # (B, Th, d_model)
        a_hist_labels,   # (B, Th)   <-- NEW
        q_text_emb,      # (B, d_model)
        task_labels      # (B,)
    ):
        B, Th, _ = e_hist.shape
        device = e_hist.device

        xL = torch.zeros(B, self.d_state, device=device)
        xS = torch.zeros(B, self.d_state, device=device)
        xE = torch.zeros(B, self.d_state, device=device)

        A_L = self.make_A(self.Q_L, torch.exp(self.Lambda_L))

        eps_S = torch.sigmoid(self.tau_S)
        k = torch.log((1 - eps_S) / eps_S) / torch.abs(self.Lambda_S)
        A_S = self.make_A(self.Q_S, torch.sigmoid(-k * self.Lambda_S))

        eta = (torch.exp(-1.0 / self.tau_E) - 1.0) / self.Lambda_E
        A_E = self.make_A(self.Q_E, 1.0 + eta * self.Lambda_E)

        loss_action = 0.0
        loss_node = 0.0

        # =====================================================
        # HISTORY LOOP WITH PER-STEP LOSSES
        # =====================================================
        for t in range(Th):
            e_t = e_hist[:, t]
            a_t = a_hist_emb[:, t]

            # -------- pre-step state (for action pred) --------
            s_pre = torch.cat([xL, xS, xE], dim=-1)
            act_logits = self.ActionHead(s_pre)
            loss_action = loss_action + self.ce(act_logits, a_hist_labels[:, t])

            # -------- state update --------
            mood_vec = self.MLP_mood(a_t)
            mood = torch.sigmoid(self.W_a(mood_vec))

            u_t = torch.bmm(
                (self.B.unsqueeze(0) * mood.unsqueeze(2)),
                e_t.unsqueeze(2)
            ).squeeze(2)

            xL = torch.matmul(xL, A_L.T) + u_t
            xS = torch.matmul(xS, A_S.T) + u_t
            xE = torch.matmul(xE, A_E.T) + u_t

            # -------- post-step node pred --------
            yL = self.Mixer_L(xL)
            yS = self.Mixer_S(xS)
            yE = self.Mixer_E(xE)
            w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)

            h_step = self.Mixer_Fuse(
                w[:, 0:1] * yL +
                w[:, 1:2] * yS +
                w[:, 2:3] * yE
            )

            e_pred = self.NodeHead(h_step)
            loss_node = loss_node + self.mse(e_pred, e_t).mean(dim=-1).mean()

        loss_action = loss_action / Th
        loss_node = loss_node / Th

        # =====================================================
        # FINAL FUSION (UNCHANGED)
        # =====================================================
        yL = self.Mixer_L(xL)
        yS = self.Mixer_S(xS)
        yE = self.Mixer_E(xE)

        w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)
        h_hist = self.Mixer_Fuse(
            w[:, 0:1] * yL +
            w[:, 1:2] * yS +
            w[:, 2:3] * yE
        )

        q_emb = self.Embed_Q(q_text_emb)
        z_switch = torch.cat([h_hist, q_emb], dim=-1)
        task_logits = self.W_task(z_switch)
        P_task = F.softmax(task_logits, dim=-1)
        loss_task = self.ce(task_logits, task_labels)

        h_hyp = F.relu(self.W_hyp(z_switch))

        if self.known_task:
            theta = self.W_theta[int(task_labels[0].item())](h_hyp)
            g, b = theta.chunk(2, dim=-1)
            h_final = torch.sigmoid(g) * h_hist + b
        else:
            h_final = 0.0
            for i in range(len(self.W_theta)):
                theta_i = self.W_theta[i](h_hyp)
                g_i, b_i = theta_i.chunk(2, dim=-1)
                h_final = h_final + P_task[:, i:i+1] * (torch.sigmoid(g_i) * h_hist + b_i)

        # =====================================================
        # FINAL LOSS (ALL CONTRIBUTORS)
        # =====================================================
        loss_total = loss_task + loss_action + loss_node

        return {
            "h_hist": h_hist,
            "h_final": h_final,
            "task_logits": task_logits,
            "P_task": P_task,
            "loss_task": loss_task,
            "loss_action": loss_action,
            "loss_node": loss_node,
            "loss_total": loss_total
        }


In [71]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5Tokenizer, T5ForConditionalGeneration

# --------------------------------------------------
# User head (unchanged)
# --------------------------------------------------

class UserHead(nn.Module):
    def __init__(self, d_state, n_users):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.ReLU(),
            nn.Linear(d_state, n_users)
        )
        self.ce = nn.CrossEntropyLoss()

    def forward(self, h_final, user_labels):
        logits = self.mlp(h_final)
        loss = self.ce(logits, user_labels)
        return logits, loss


# --------------------------------------------------
# Item head (unchanged)
# --------------------------------------------------

class ItemHeadMLP(nn.Module):
    def __init__(self, d_state, n_items):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.ReLU(),
            nn.Linear(d_state, n_items)
        )
        self.ce = nn.CrossEntropyLoss()

    def forward(self, h_final, item_labels):
        logits = self.mlp(h_final)
        loss = self.ce(logits, item_labels)
        return logits, loss


# --------------------------------------------------
# Lens Gate (unchanged)
# --------------------------------------------------

class LensGate(nn.Module):
    def __init__(self, d_state, d_qtext):
        super().__init__()
        self.Wq = nn.Linear(d_qtext, d_state)
        self.Wfilm = nn.Linear(d_state, 2 * d_state)
        self.Wg = nn.Linear(2 * d_state, d_state)

    def forward(self, h_final, q_text_emb):
        q0 = self.Wq(q_text_emb)
        gamma, beta = self.Wfilm(h_final).chunk(2, dim=-1)
        q_lens = (1.0 + gamma) * q0 + beta
        g = torch.sigmoid(self.Wg(torch.cat([h_final, q_lens], dim=-1)))
        return g * q_lens + (1.0 - g) * q0


# ============================================================
# T5Generator (teacher forcing + generate)
# Fixes your prior encoder_outputs tuple error.
# ============================================================

import torch
import torch.nn as nn
from transformers import T5Tokenizer, T5ForConditionalGeneration

class T5Generator(nn.Module):
    def __init__(self, d_state, device, model_name="t5-small"):
        super().__init__()
        self.device = device
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.t5 = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

        # freeze T5
        for p in self.t5.parameters():
            p.requires_grad = False

        self.d_t5 = self.t5.config.d_model
        self.hist_proj = nn.Linear(d_state, self.d_t5)

    def forward(self, h_final, query_text, target_text=None, reduction="mean"):
        """
        Training: target_text=list[str] -> returns scalar loss (mean) OR per-sample loss if reduction="none"
        Inference: target_text=None -> returns list[str] generations
        """
        # tokenize query text
        enc = self.tokenizer(
            query_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(self.device)

        # embed input ids
        with torch.no_grad():
            emb = self.t5.encoder.embed_tokens(enc.input_ids)  # (B,T,512)

        # add projected history vector to every encoder token
        h_proj = self.hist_proj(h_final).unsqueeze(1)  # (B,1,512)
        emb = emb + h_proj

        # ---------------- TRAIN ----------------
        if target_text is not None:
            tgt = self.tokenizer(
                target_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128
            ).to(self.device)

            out = self.t5(
                inputs_embeds=emb,
                attention_mask=enc.attention_mask,
                labels=tgt.input_ids
            )

            if reduction == "mean":
                return out.loss

            # reduction="none": compute per-sample CE by shifting manually
            # HuggingFace loss is averaged; we need per-sample losses for masking.
            # We'll compute token-level NLL ourselves.
            logits = out.logits  # (B, L, V)
            labels = tgt.input_ids  # (B, L)
            # mask pad tokens
            pad = self.tokenizer.pad_token_id
            label_mask = (labels != pad).float()

            # shift for teacher forcing CE: predict labels[:,1:] from logits[:,:-1]
            logits_s = logits[:, :-1, :].contiguous()
            labels_s = labels[:, 1:].contiguous()
            mask_s   = label_mask[:, 1:].contiguous()

            logp = torch.log_softmax(logits_s, dim=-1)
            nll = -logp.gather(-1, labels_s.unsqueeze(-1)).squeeze(-1)  # (B,L-1)
            nll = nll * mask_s
            per = nll.sum(dim=-1) / mask_s.sum(dim=-1).clamp_min(1.0)   # (B,)
            return per

        # ---------------- INFER ----------------
        gen = self.t5.generate(
            inputs_embeds=emb,
            attention_mask=enc.attention_mask,
            max_length=128,
            num_beams=4
        )
        return self.tokenizer.batch_decode(gen, skip_special_tokens=True)


In [76]:
# ============================================================
# CELL 18 — Full Multi-Task Model (CONSISTENT VERSION)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class UniHisMultiTask(nn.Module):
    def __init__(
        self,
        n_actions,
        d_model,
        d_state,
        n_tasks,
        n_users,
        n_items,
        device,
        known_task=False
    ):
        super().__init__()

        self.device = device
        self.d_model = d_model
        self.d_state = d_state
        self.n_tasks = n_tasks

        # Action embedding
        self.action_proj = nn.Linear(n_actions, d_model)

        # Shared history encoder (UPDATED)
        self.encoder = UniHisSSMEncoder(
            d_model=d_model,
            d_state=d_state,
            n_tasks=n_tasks,
            n_actions=n_actions,
            known_task=known_task
        )

        # Heads
        self.user_head = UserHead(d_state, n_users) if n_users > 0 else None
        self.item_head = ItemHeadMLP(d_state, n_items) if n_items > 0 else None

        # Frozen-T5 generator
        self.generator = T5Generator(d_state=d_state, device=device)

    def forward(self, batch):
        """
        batch contains:
          e_hist                 (B,Th,768)
          action_onehot_hist     (B,Th,n_actions)
          action_labels          (B,Th)
          q_text_emb             (B,768)
          task_labels            (B,)
          user_mask, item_mask, gen_mask
          user_labels, item_labels
          query_text_str, target_text_str
        """

        # ----- Encode history -----
        e_hist = batch["e_hist"]
        a_hist = self.action_proj(batch["action_onehot_hist"])
        q_emb  = batch["q_text_emb"]

        enc = self.encoder(
            e_hist=e_hist,
            a_hist_emb=a_hist,
            a_hist_labels=batch["action_labels"],
            q_text_emb=q_emb,
            task_labels=batch["task_labels"]
        )

        h_final     = enc["h_final"]
        loss_task   = enc["loss_task"]
        loss_action = enc["loss_action"]
        loss_node   = enc["loss_node"]

        # ----- User head -----
        user_logits = None
        loss_user = torch.zeros((), device=h_final.device)
        if self.user_head is not None:
            user_logits = self.user_head.mlp(h_final)
            m = batch["user_mask"]
            if m.sum().item() > 0:
                ce = F.cross_entropy(
                    user_logits,
                    batch["user_labels"],
                    reduction="none"
                )
                loss_user = (ce * m).sum() / m.sum().clamp_min(1.0)

        # ----- Item head -----
        item_logits = None
        loss_item = torch.zeros((), device=h_final.device)
        if self.item_head is not None:
            item_logits = self.item_head.mlp(h_final)
            m = batch["item_mask"]
            if m.sum().item() > 0:
                ce = F.cross_entropy(
                    item_logits,
                    batch["item_labels"],
                    reduction="none"
                )
                loss_item = (ce * m).sum() / m.sum().clamp_min(1.0)

        # ----- Generator loss (teacher forcing) -----
        loss_gen = torch.zeros((), device=h_final.device)
        m = batch["gen_mask"]
        if m.sum().item() > 0:
            per_loss = self.generator(
                h_final=h_final,
                query_text=batch["query_text_str"],
                target_text=batch["target_text_str"],
                reduction="none"
            )
            loss_gen = (per_loss * m).sum() / m.sum().clamp_min(1.0)

        # ----- FINAL LOSS (ALL CONTRIBUTORS) -----
        loss_total = (
            loss_task +
            loss_action +
            loss_node +
            loss_user +
            loss_item +
            loss_gen
        )

        return {
            "h_final": h_final,
            "task_logits": enc["task_logits"],
            "user_logits": user_logits,
            "item_logits": item_logits,

            "loss_task": loss_task,
            "loss_action": loss_action,
            "loss_node": loss_node,
            "loss_user": loss_user,
            "loss_item": loss_item,
            "loss_gen":  loss_gen,
            "loss_total": loss_total
        }


In [ ]:
# ============================================================
# CELL 19 — Instantiate Model + Optimizer
# Frozen-T5 conditional generator version
# ============================================================

import torch

# --- REQUIRED: provide d_state explicitly ---
D_STATE = 64

model = UniHisMultiTask(
    n_actions=len(actions2idx),
    d_model=768,
    d_state=D_STATE,
    n_tasks=n_tasks,
    n_users=len(user2idx),
    n_items=len(item2idx),
    device=DEVICE,
    known_task=False
).to(DEVICE)

# Only train what requires_grad (T5 is frozen)
optim = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4
)

print("Model on:", DEVICE)
print("n_actions:", len(actions2idx))
print("n_users:", len(user2idx))
print("n_items:", len(item2idx))
print("n_tasks:", n_tasks)

def count_parameters(m, only_trainable=False):
    if only_trainable:
        return sum(p.numel() for p in m.parameters() if p.requires_grad)
    return sum(p.numel() for p in m.parameters())

total_params = count_parameters(model, only_trainable=False)
trainable_params = count_parameters(model, only_trainable=True)

print("\n========== PARAMETER REPORT ==========")
print("Total parameters     :", f"{total_params:,}")
print("Trainable parameters :", f"{trainable_params:,}")
print("Trainable fraction   :", f"{trainable_params / max(1,total_params):.6f}")


In [ ]:
# ============================================================
# TEXT LOOKUP TABLES (RUN ONCE)
# ============================================================

# dialogue: dialogue_id -> dialogue_text
dialog_map = dialog_df.set_index("dialogue_id")["dialogue_text"].astype(str)

# responses: response_id -> response_text
resp_map = resp_df.set_index("response_id")["response_text"].astype(str)

# summaries: SummID -> Summary
summ_map = summ_df.set_index("SummID")["Summary"].astype(str)

# documents: NewsID -> NewsBody
docs_map = docs_df.set_index("NewsID")["NewsBody"].astype(str)

print("Loaded text maps:")
print("dialog:", len(dialog_map))
print("resp  :", len(resp_map))
print("summ  :", len(summ_map))
print("docs  :", len(docs_map))


In [92]:
# ============================================================
# CELL 20 — Batch Builder (row -> tensors)
# Frozen-T5 conditional generation (CORRECTED)
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# ASSUMPTIONS (must already exist in runtime)
# ------------------------------------------------------------
# dialog_map : Dict[str, str]
# docs_map   : Dict[str, str]
# resp_map   : Dict[str, str]
# summ_map   : Dict[str, str]
#
# emb_resolver : Dict[str, np.ndarray]  # id -> (768,)
# actions2idx  : Dict[str, int]
# user2idx     : Dict[str, int]
# item2idx     : Dict[str, int]
#
# TASK_NEXT_USER, TASK_NEXT_ITEM
# KP_TASKS
# MAX_T
# parse_serialized_sequence
# normalize_id
# ------------------------------------------------------------

# ============================================================
# TEXT RESOLVERS
# ============================================================

def resolve_query_text(qid: str) -> str:
    qid = str(qid)

    if qid.startswith("dl-"):
        return dialog_map.get(qid, "")
    if qid.startswith("d-"):
        return docs_map.get(qid, "")

    return ""


def resolve_target_text(tid: str) -> str:
    tid = str(tid)

    if tid.startswith("r-"):
        return resp_map.get(tid, "")
    if tid.startswith("s-"):
        return summ_map.get(tid, "")

    return ""


# ============================================================
# ROW → TENSORS
# ============================================================

def build_row_tensors(row, device):
    # ---------------- task ----------------
    task_name = str(row["task"]).strip()
    task_id   = int(row["task_id"])

    task_labels = torch.tensor(
        [task_id], device=device, dtype=torch.long
    )  # (1,)

    # ---------------- sequences ----------------
    actions = parse_serialized_sequence(row["interactions"])
    nodes   = parse_serialized_sequence(row["text_nodes"])

    if len(actions) == 0 or len(nodes) == 0:
        raise ValueError("empty sequence")
    if len(actions) != len(nodes):
        raise ValueError("actions/nodes length mismatch")
    if len(actions) < 2:
        raise ValueError("sequence too short (need >=2)")

    if len(actions) > MAX_T:
        actions = actions[-MAX_T:]
        nodes   = nodes[-MAX_T:]

    actions_hist = actions[:-2]
    nodes_hist   = nodes[:-2]
    Th = len(actions_hist)

    # ---------------- action one-hot (history) ----------------
    a_ids = [actions2idx[a] for a in actions_hist]
    a_ids = torch.tensor(
        a_ids, device=device, dtype=torch.long
    ).unsqueeze(0)                       # (1, Th)

    action_onehot_hist = F.one_hot(
        a_ids, num_classes=len(actions2idx)
    ).float()                            # (1, Th, A)

    # ---------------- node embeddings (history) ----------------
    nodes_hist = [normalize_id(n) for n in nodes_hist]

    e_hist_np = np.stack(
        [emb_resolver.get(n) for n in nodes_hist],
        axis=0
    )                                    # (Th, 768)

    e_hist = torch.tensor(
        e_hist_np, device=device, dtype=torch.float32
    ).unsqueeze(0)                       # (1, Th, 768)

    # ---------------- query embedding (adapter input) ----------------
    qid = normalize_id(row["query_text_id"])

    q_text_emb = torch.tensor(
        emb_resolver.get(qid), device=device, dtype=torch.float32
    ).unsqueeze(0)                       # (1, 768)

    # ---------------- raw query text (T5 input) ----------------
    q_text_str = resolve_query_text(qid)

    # ---------------- target text (teacher forcing) ----------------
    tid = normalize_id(row.get("target_text_id", ""))
    target_text_str = resolve_target_text(tid)

    # ---------------- masks ----------------
    user_mask = torch.zeros(1, device=device)
    item_mask = torch.zeros(1, device=device)
    gen_mask  = torch.zeros(1, device=device)

    # ---------------- labels ----------------
    user_labels = torch.zeros(1, device=device, dtype=torch.long)
    item_labels = torch.zeros(1, device=device, dtype=torch.long)

    # ---- Next-user prediction ----
    if task_name == TASK_NEXT_USER:
        uid = str(row["userid"]).strip()
        if uid not in user2idx:
            raise ValueError(f"userid not in vocab: {uid}")

        user_labels[0] = user2idx[uid]
        user_mask[0]   = 1.0

    # ---- Next-item prediction (FIXED) ----
    if task_name == TASK_NEXT_ITEM:
        it = normalize_id(row["query_text_id"])
        if it not in item2idx:
            raise ValueError(f"item not in vocab: {it}")
        item_labels[0] = item2idx[it]
        item_mask[0] = 1.0

    # ---- Generation tasks (dialogue / summarization) ----
    if task_name in KP_TASKS:
        if len(q_text_str) > 0 and len(target_text_str) > 0:
            gen_mask[0] = 1.0


    # ----- action labels (broadcast to Th for per-step loss) -----
    action_target = actions[-2]
    if action_target not in actions2idx:
        raise ValueError(f"action not in vocab: {action_target}")

    # encoder expects (B, Th)
    action_labels = torch.full(
        (1, Th),
        actions2idx[action_target],
        device=device,
        dtype=torch.long
    )

    # ---------------- output ----------------
    return {
        "task_name": task_name,

        "e_hist": e_hist,
        "action_onehot_hist": action_onehot_hist,
        "q_text_emb": q_text_emb,

        # T5 expects List[str] (batch size = 1)
        "query_text_str":  [q_text_str],
        "target_text_str": [target_text_str],

        "task_labels": task_labels,
        "action_labels": action_labels,

        "user_labels": user_labels,
        "item_labels": item_labels,

        "user_mask": user_mask,
        "item_mask": item_mask,
        "gen_mask":  gen_mask,
    }


In [ ]:
# ============================================================
# One-Sample Per Task: Forward + Inference Sanity Check (FIXED)
# Tasks:
#   - next_user
#   - next_item
#   - dialogue_generation
#   - personalized_summarization
# ============================================================

import pandas as pd
import torch

pd.set_option("display.max_colwidth", 160)

model.eval()

# ------------------------------------------------------------
# Select exactly:
#   1 × next_user
#   1 × next_item
#   2 × generation tasks (from KP_TASKS)
# ------------------------------------------------------------

TASKS_WANTED = [TASK_NEXT_USER, TASK_NEXT_ITEM]

gen_tasks_wanted = []
if isinstance(KP_TASKS, (list, tuple, set)):
    KP_TASKS_LIST = list(KP_TASKS)

    for preferred in ["dialogue_generation",'personalized_summarization']:
        if preferred in KP_TASKS_LIST:
            gen_tasks_wanted.append(preferred)

    for t in KP_TASKS_LIST:
        if t not in gen_tasks_wanted:
            gen_tasks_wanted.append(t)

    gen_tasks_wanted = gen_tasks_wanted[:2]

TASKS_WANTED = TASKS_WANTED + gen_tasks_wanted

# ------------------------------------------------------------
# Pick one row per task
# ------------------------------------------------------------

samples = {}
for _, row in train_df.iterrows():
    t = str(row["task"]).strip()
    if t in TASKS_WANTED and t not in samples:
        samples[t] = row
    if len(samples) == len(TASKS_WANTED):
        break

print("Requested tasks:", TASKS_WANTED)
print("Found samples   :", list(samples.keys()))

missing = [t for t in TASKS_WANTED if t not in samples]
if missing:
    print("Missing tasks in train_df:", missing)

# ------------------------------------------------------------
# Forward + decode
# ------------------------------------------------------------

rows = []

with torch.no_grad():
    for task, row in samples.items():
        batch = build_row_tensors(row, DEVICE)
        out = model(batch)

        rec = {
            "task": task,
            "query_id": normalize_id(row.get("query_text_id", "")),
            "target_id": normalize_id(row.get("target_text_id", "")),
            "loss_task": float(out["loss_task"].item()),
            "loss_user": float(out["loss_user"].item()),
            "loss_item": float(out["loss_item"].item()),
            "loss_gen":  float(out["loss_gen"].item()),
        }

        # ---------------- USER ----------------
        if task == TASK_NEXT_USER:
            true_u = str(row.get("userid", "")).strip()
            pred_u_idx = int(torch.argmax(out["user_logits"], dim=-1).item())

            rec["true_user"] = true_u
            rec["pred_user"] = idx2user.get(pred_u_idx, str(pred_u_idx))
        else:
            rec["true_user"] = "-"
            rec["pred_user"] = "-"

        # ---------------- ITEM (FIXED) ----------------
        if task == TASK_NEXT_ITEM:
            nodes = parse_serialized_sequence(row["text_nodes"])
            true_item_id = normalize_id(nodes[-2])

            pred_i_idx = int(torch.argmax(out["item_logits"], dim=-1).item())

            rec["true_item"] = true_item_id
            rec["pred_item"] = idx2item.get(pred_i_idx, str(pred_i_idx))
        else:
            rec["true_item"] = "-"
            rec["pred_item"] = "-"

        # ---------------- TEXT ----------------
        rec["query_text"] = batch["query_text_str"][0]
        rec["true_text"]  = batch["target_text_str"][0]

        # ---------------- AUTOREGRESSIVE GENERATION ----------------
        if batch["gen_mask"].sum().item() > 0:
            gen_text = model.generator(
                h_final=out["h_final"],
                query_text=batch["query_text_str"],
                target_text=None
            )[0]
            rec["pred_text"] = gen_text
        else:
            rec["pred_text"] = "-"

        rows.append(rec)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

df_debug = pd.DataFrame(rows)
display(df_debug)


In [ ]:
# ============================================================
# CELL 21 — TRAINING LOOP (STEP CKPT + ROW RESUME + TQDM LOSSES)
# ============================================================

import os
import torch
from torch.optim import AdamW
from collections import defaultdict
from tqdm import tqdm

# ---------------- device ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

# ---------------- hyperparams ----------------
EPOCHS = 5
LR = 2e-4
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 15000

# ---------------- checkpoint config ----------------
CKPT_DIR = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def ckpt_path(epoch, step):
    return os.path.join(CKPT_DIR, f"ckpt_e{epoch}_s{step}.pt")

# ---------------- optimizer ----------------
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=1e-4
)

# ============================================================
# RESUME FROM LATEST CHECKPOINT (ROW-LEVEL)
# ============================================================

start_epoch = 0
start_row = 0
global_step = 0

ckpts = [f for f in os.listdir(CKPT_DIR) if f.endswith(".pt")]
if ckpts:
    ckpts = sorted(
        ckpts,
        key=lambda x: (
            int(x.split("_e")[1].split("_")[0]),
            int(x.split("_s")[1].split(".")[0])
        )
    )
    last_ckpt = ckpts[-1]
    ckpt = torch.load(os.path.join(CKPT_DIR, last_ckpt), map_location=device)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])

    start_epoch = ckpt["epoch"]
    start_row   = ckpt["row"]
    global_step = ckpt["global_step"]

    print(f"✔ Resumed from {last_ckpt}")
    print(f"✔ epoch={start_epoch}, row={start_row}, step={global_step}")

# ============================================================
# TRAINING
# ============================================================

for epoch in range(start_epoch, EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    running = defaultdict(float)
    step_in_epoch = 0

    row_range = range(start_row, len(train_df)) if epoch == start_epoch else range(len(train_df))
    pbar = tqdm(row_range, total=len(row_range))

    for i in pbar:
        row = train_df.iloc[i]

        try:
            batch = build_row_tensors(row, device)
        except Exception:
            continue

        optimizer.zero_grad(set_to_none=True)

        out = model(batch)
        loss = out["loss_total"]

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        # ---------------- running stats ----------------
        step_in_epoch += 1
        global_step += 1

        for k in [
            "loss_total",
            "loss_gen",
            "loss_item",
            "loss_user",
        ]:
            running[k] += float(out[k].detach().item())

        # ---------------- tqdm live display ----------------
        pbar.set_postfix({
            "L":  f"{running['loss_total']/step_in_epoch:.4f}",
            "Lg": f"{running['loss_gen']/step_in_epoch:.4f}",
            "Li": f"{running['loss_item']/step_in_epoch:.4f}",
            "Lu": f"{running['loss_user']/step_in_epoch:.4f}",
        })

        # ---------------- STEP CHECKPOINT ----------------
        if global_step % SAVE_EVERY_STEPS == 0:
            torch.save(
                {
                    "epoch": epoch,
                    "row": i + 1,  # resume from NEXT row
                    "global_step": global_step,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                },
                ckpt_path(epoch, global_step)
            )

    # reset row offset after resumed epoch
    start_row = 0

# ============================================================
# END TRAINING
# ============================================================


In [ ]:
# ============================================================
# Decoding helpers — MUST run before inference
# ============================================================

import torch
import numpy as np

@torch.no_grad()
def topk_from_logits(logits, idx2tok, k=10):
    probs = torch.softmax(logits, dim=-1)
    k = min(k, probs.size(-1))
    topv, topi = torch.topk(probs, k=k, dim=-1)
    ids = topi[0].tolist()
    vals = topv[0].tolist()
    return [(idx2tok[i], float(v)) for i, v in zip(ids, vals)]

@torch.no_grad()
def decode_kp_logits(kp_logits, idx2kp, topk=15, threshold=0.03):
    scores = torch.sigmoid(kp_logits)
    out_all = []
    for b in range(scores.size(0)):
        s = scores[b]
        keep = s >= threshold
        idxs = torch.nonzero(keep).squeeze(-1)
        if idxs.numel() == 0:
            out_all.append([])
            continue
        vals = s[idxs]
        k = min(topk, vals.numel())
        topv, topj = torch.topk(vals, k)
        phrases = [(idx2kp[int(idxs[topj[t]])], float(topv[t].item())) for t in range(k)]
        out_all.append(phrases)
    return out_all

# ============================================================
# CELL 23 — Inference on test_df (task-wise outputs)
# ============================================================

CKPT_PATH = os.path.join(CKPT_DIR, "step_2_12000.pt")

if CKPT_PATH is not None:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    model.to(DEVICE)

model.eval()

records = []
TOPK = 10

with torch.no_grad():
    for i in tqdm(range(len(test_df[:50])), desc="Inference"):
        row = test_df.iloc[i]

        try:
            batch = build_row_tensors(row, DEVICE)
        except Exception:
            continue

        out = model(batch)
        task_name = batch["task_name"]

        rec = {
            "task": task_name,
            "userid_raw": str(row.get("userid", "")),
            "query_text_id": normalize_id(row.get("query_text_id", "")),
            "target_text_id": normalize_id(row.get("target_text_id", "")),
        }

        # ---- task prediction ----
        rec["pred_task_top10"] = topk_from_logits(
            out["task_logits"], idx2task, k=min(10, len(idx2task))
        )

        # ---- user prediction ----
        if task_name == TASK_NEXT_USER and out["user_logits"] is not None:
            true_uid = str(row["userid"]).strip()
            rec["true_userid"] = true_uid
            rec["pred_user_top10"] = topk_from_logits(
                out["user_logits"], idx2user, k=min(TOPK, len(idx2user))
            )
        else:
            rec["true_userid"] = None
            rec["pred_user_top10"] = []

        # ---- item prediction ----
        if task_name == TASK_NEXT_ITEM and out["item_logits"] is not None:
            true_item = normalize_id(row["query_text_id"])
            rec["true_item"] = true_item
            rec["pred_item_top10"] = topk_from_logits(
                out["item_logits"], idx2item, k=min(TOPK, len(idx2item))
            )
        else:
            rec["true_item"] = None
            rec["pred_item_top10"] = []

        # ---- KP tasks ----
        if task_name in KP_TASKS and out["kp_logits"] is not None:
            kp_id = get_kp_supervision_id(row)

            if kp_id is None:
                gold_kps = []
            else:
                _, gold_kps = build_kp_target_from_id(
                    kp_id, id2phrases, kp2idx, DEVICE
                )

            rec["gold_keyphrases"] = gold_kps

            scored = decode_kp_logits(
                out["kp_logits"], idx2kp, topk=TOPK, threshold=0.03
            )[0]

            rec["pred_keyphrases_scored"] = scored
            rec["pred_keyphrases"] = [kp for kp, _ in scored]
        else:
            rec["gold_keyphrases"] = []
            rec["pred_keyphrases"] = []

        # ---- losses ----
        rec["loss_task"] = float(out["loss_task"].item())
        rec["loss_user"] = float(out["loss_user"].item())
        rec["loss_item"] = float(out["loss_item"].item())
        rec["loss_kp"]   = float(out["loss_kp"].item())

        records.append(rec)

results_df = pd.DataFrame(records)
print("Inference records:", len(results_df))
results_df.head(3)


In [ ]:
# ============================================================
# CELL A — Ranking Metrics: MRR, nDCG@K
# Purpose:
#   - Compute MRR, nDCG@5, nDCG@10 for user & item prediction
# Inputs:
#   - results_df
# Outputs:
#   - metric dicts
# ============================================================

import numpy as np

def reciprocal_rank(true_id, ranked_ids):
    for i, rid in enumerate(ranked_ids):
        if rid == true_id:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(true_id, ranked_ids, k):
    if true_id not in ranked_ids[:k]:
        return 0.0
    rank = ranked_ids.index(true_id)
    return 1.0 / np.log2(rank + 2)

def eval_ranking(df, task_name, true_col, pred_col):
    mrr = []
    ndcg20 = []
    ndcg100 = []

    for _, r in df[df["task"] == task_name].iterrows():
        true = r[true_col]
        preds = [x[0] for x in r[pred_col]]

        if true is None or len(preds) == 0:
            continue

        mrr.append(reciprocal_rank(true, preds))
        ndcg20.append(ndcg_at_k(true, preds, 20))
        ndcg100.append(ndcg_at_k(true, preds, 100))

    return {
        "MRR": np.mean(mrr) if mrr else 0.0,
        "nDCG@20": np.mean(ndcg20) if ndcg20 else 0.0,
        "nDCG@100": np.mean(ndcg100) if ndcg100 else 0.0,
        "N": len(mrr)
    }

# ============================================================
# CELL B — User & Item Prediction Evaluation
# ============================================================

user_metrics = eval_ranking(
    results_df,
    task_name=TASK_NEXT_USER,
    true_col="true_userid",
    pred_col="pred_user_top10"
)

item_metrics = eval_ranking(
    results_df,
    task_name=TASK_NEXT_ITEM,
    true_col="true_item",
    pred_col="pred_item_top10"
)

print("\n=== USER PREDICTION ===")
for k, v in user_metrics.items():
    print(f"{k:10s}: {v}")

print("\n=== ITEM PREDICTION ===")
for k, v in item_metrics.items():
    print(f"{k:10s}: {v}")


In [ ]:
# ============================================================
# CELL C′ — Keyphrase Metrics (NO external deps)
# Purpose:
#   - Precision@K
#   - Recall@K
#   - ROUGE-1 / ROUGE-2 / ROUGE-L Recall
# Inputs:
#   - results_df
# Outputs:
#   - averaged metrics
# ============================================================

import numpy as np

# ---------- helpers ----------
def ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))

def rouge_n_recall(pred_tokens, ref_tokens, n):
    if len(ref_tokens) < n:
        return 0.0
    ref_ngrams = ngrams(ref_tokens, n)
    pred_ngrams = ngrams(pred_tokens, n)
    if not ref_ngrams:
        return 0.0
    ref_count = {}
    for ng in ref_ngrams:
        ref_count[ng] = ref_count.get(ng, 0) + 1
    hit = 0
    for ng in pred_ngrams:
        if ref_count.get(ng, 0) > 0:
            hit += 1
            ref_count[ng] -= 1
    return hit / len(ref_ngrams)

def lcs_length(x, y):
    # classic DP LCS
    dp = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(len(x)):
        for j in range(len(y)):
            if x[i] == y[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])
    return dp[-1][-1]

def rouge_l_recall(pred_tokens, ref_tokens):
    if not ref_tokens:
        return 0.0
    lcs = lcs_length(pred_tokens, ref_tokens)
    return lcs / len(ref_tokens)

def precision_recall_at_k(pred, gold, k):
    pred_k = pred[:k]
    if not pred_k:
        return 0.0, 0.0
    gold_set = set(gold)
    hit = sum(1 for p in pred_k if p in gold_set)
    precision = hit / len(pred_k)
    recall = hit / max(len(gold), 1)
    return precision, recall

# ---------- evaluation ----------
def eval_keyphrases(results_df, k=10):
    prec, rec = [], []
    r1, r2, rL = [], [], []

    for _, r in results_df[results_df["task"].isin(KP_TASKS)].iterrows():
        gold = r["gold_keyphrases"]
        pred = r["pred_keyphrases"]

        if not gold or not pred:
            continue

        p, r_ = precision_recall_at_k(pred, gold, k)
        prec.append(p)
        rec.append(r_)

        gold_tokens = " ".join(gold).split()
        pred_tokens = " ".join(pred[:k]).split()

        r1.append(rouge_n_recall(pred_tokens, gold_tokens, n=1))
        r2.append(rouge_n_recall(pred_tokens, gold_tokens, n=2))
        rL.append(rouge_l_recall(pred_tokens, gold_tokens))

    return {
        f"Precision@{k}": np.mean(prec) if prec else 0.0,
        f"Recall@{k}": np.mean(rec) if rec else 0.0,
        "ROUGE-1 Recall": np.mean(r1) if r1 else 0.0,
        "ROUGE-2 Recall": np.mean(r2) if r2 else 0.0,
        "ROUGE-L Recall": np.mean(rL) if rL else 0.0,
        "N": len(prec)
    }
kp_metrics_5 = eval_keyphrases(results_df, k=5)
kp_metrics_10 = eval_keyphrases(results_df, k=10)

print("\n=== KEYPHRASE METRICS @5 ===")
for k, v in kp_metrics_5.items():
    print(f"{k:18s}: {v}")

print("\n=== KEYPHRASE METRICS @10 ===")
for k, v in kp_metrics_10.items():
    print(f"{k:18s}: {v}")


In [ ]:
results_df